# 109 — Delta-ML Family Ensemble Blend

**Motivation:** We have 5+ delta-ML variants (nb76, nb97, nb104, nb105, nb106, nb107, nb108). Each captures different aspects of the similarity-activity landscape. A meta-learner over just this delta family should find the optimal combination.

**Strategy:**
1. Load OOF arrays for all available delta-ML models
2. Nested-CV ElasticNet blending (same protocol as nb96 grand ensemble)
3. Also compute: simple average and rank-weighted average
4. Report OOF RAE for all strategies
5. Save best as oof_delta_ensemble_blend.npy

**Expected gain:** Each delta variant has correlated errors for hard/fallback compounds; the meta-learner will down-weight variants that over-fit to noisy low-similarity pairs.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)

In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")

Train 4,139  Test 513  Cliffs 0


In [4]:
# --- Load delta-ML family OOF arrays ---
# Candidate delta models in priority order
DELTA_CANDIDATES = [
    "delta_ml",              # nb76: single-template delta
    "multi_template_delta",  # nb97: multi-template delta
    "delta_similarity_tiers",# nb104: 3-tier tiered delta
    "delta_uncertainty",     # nb105: uncertainty-weighted delta
    "reverse_delta_ml",      # nb106: reverse/transductive delta
    "delta_5tiers",          # nb107: 5-tier delta (may not exist yet)
    "delta_loso",            # nb108: scaffold-LOSO delta (may not exist yet)
    "consensus_delta_ml",    # nb99: consensus delta (may exist)
    "delta_chemprop_cpu",    # nb103: chemprop delta
]

oofs, tes, names = [], [], []
for name in DELTA_CANDIDATES:
    fp = DATA_PROCESSED / f"oof_{name}.npy"
    te_fp = DATA_PROCESSED / f"te_oof_{name}.npy"
    if not fp.exists():
        print(f"  SKIP {name}: OOF not found")
        continue
    try:
        arr = np.load(fp)
        if arr.ndim > 1: arr = arr[:,0]
        if len(arr) != len(y_tr):
            print(f"  SKIP {name}: wrong length {len(arr)}"); continue
        te_v = np.load(te_fp) if te_fp.exists() else None
        if te_v is None or len(te_v) != 513:
            print(f"  SKIP {name}: no valid te_oof"); continue
        if te_v.ndim > 1: te_v = te_v[:,0]
        arr[~np.isfinite(arr)] = y_tr.mean()
        te_v[~np.isfinite(te_v)] = float(np.nanmean(te_v))
        individual_rae = rae(y_tr, arr)
        oofs.append(arr); tes.append(te_v); names.append(name)
        print(f"  LOAD {name}: OOF RAE = {individual_rae:.4f}")
    except Exception as e:
        print(f"  ERROR {name}: {e}")

print(f"\nDelta family: {len(names)} models loaded")
print(f"Models: {names}")

if len(oofs) < 2:
    raise RuntimeError("Need at least 2 delta OOF arrays to blend. Run nb97-nb108 first.")

OOF_stack = np.column_stack(oofs)
TE_stack  = np.column_stack(tes)
print(f"Stack shape: {OOF_stack.shape}")

  LOAD delta_ml: OOF RAE = 0.4164
  LOAD multi_template_delta: OOF RAE = 0.3266
  LOAD delta_similarity_tiers: OOF RAE = 0.2772
  LOAD delta_uncertainty: OOF RAE = 0.3268
  LOAD reverse_delta_ml: OOF RAE = 0.3269
  LOAD delta_5tiers: OOF RAE = 0.2888
  LOAD delta_loso: OOF RAE = 0.3266
  LOAD consensus_delta_ml: OOF RAE = 0.5173
  LOAD delta_chemprop_cpu: OOF RAE = 0.5585

Delta family: 9 models loaded
Models: ['delta_ml', 'multi_template_delta', 'delta_similarity_tiers', 'delta_uncertainty', 'reverse_delta_ml', 'delta_5tiers', 'delta_loso', 'consensus_delta_ml', 'delta_chemprop_cpu']
Stack shape: (4139, 9)


In [5]:
# --- Strategy 1: Simple average ---
oof_simple_avg = OOF_stack.mean(axis=1)
m_simple = full_metrics(y_tr, oof_simple_avg, cliff_pairs, "simple_average")

# --- Strategy 2: Rank-weighted average ---
# Weight each model by 1/RAE (lower RAE -> higher weight)
individual_raes = np.array([rae(y_tr, OOF_stack[:,i]) for i in range(len(names))])
rank_weights = 1.0 / np.maximum(individual_raes, 1e-6)
rank_weights /= rank_weights.sum()
oof_rank_avg = OOF_stack @ rank_weights
m_rank = full_metrics(y_tr, oof_rank_avg, cliff_pairs, "rank_weighted")

print("\nIndividual delta model RAEs:")
for n, r, w in zip(names, individual_raes, rank_weights):
    print(f"  {n:35s}  RAE={r:.4f}  rank_w={w:.4f}")

  [simple_average] RAE=0.3577 MAE=0.3255 R2=0.8089 r=0.9051 rho=0.8741 tau=0.7135
  [rank_weighted] RAE=0.3391 MAE=0.3085 R2=0.8184 r=0.9093 rho=0.8797 tau=0.7243

Individual delta model RAEs:
  delta_ml                             RAE=0.4164  rank_w=0.0944
  multi_template_delta                 RAE=0.3266  rank_w=0.1204
  delta_similarity_tiers               RAE=0.2772  rank_w=0.1418
  delta_uncertainty                    RAE=0.3268  rank_w=0.1203
  reverse_delta_ml                     RAE=0.3269  rank_w=0.1202
  delta_5tiers                         RAE=0.2888  rank_w=0.1361
  delta_loso                           RAE=0.3266  rank_w=0.1204
  consensus_delta_ml                   RAE=0.5173  rank_w=0.0760
  delta_chemprop_cpu                   RAE=0.5585  rank_w=0.0704


In [6]:
from sklearn.linear_model import ElasticNetCV

# --- Strategy 3: Nested-CV ElasticNet blend ---
print("\n=== Nested-CV ElasticNet Blend ===", flush=True)
oof_enet = np.full(len(y_tr), np.nan)

for k, (tr_idx, va_idx) in enumerate(splits):
    # Nested: fit on all folds except the current one
    meta_tr_idx = [i for fold,(ti,_) in enumerate(splits) for i in ti if fold!=k]
    meta = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
    meta.fit(OOF_stack[meta_tr_idx], y_tr[meta_tr_idx])
    oof_enet[va_idx] = meta.predict(OOF_stack[va_idx])
    fold_rae = rae(y_tr[va_idx], oof_enet[va_idx])
    print(f"  fold {k+1}  val_RAE={fold_rae:.4f}", flush=True)

m_enet = full_metrics(y_tr, oof_enet, cliff_pairs, "enet_blend")
print(f"\nDelta Ensemble ElasticNet OOF RAE: {m_enet['RAE']:.4f}")


=== Nested-CV ElasticNet Blend ===


  fold 1  val_RAE=0.2538


  fold 2  val_RAE=0.2647


  fold 3  val_RAE=0.2955


  fold 4  val_RAE=0.2825


  fold 5  val_RAE=0.2883


  [enet_blend] RAE=0.2748 MAE=0.2501 R2=0.8539 r=0.9241 rho=0.8995 tau=0.7563

Delta Ensemble ElasticNet OOF RAE: 0.2748


In [7]:
# --- Comparison table ---
print("\n=== Delta Family Blend Comparison ===")
rows = []
for n, r in zip(names, individual_raes):
    rows.append({"method": n, "RAE": r})
rows.append({"method": "simple_average", "RAE": m_simple["RAE"]})
rows.append({"method": "rank_weighted",  "RAE": m_rank["RAE"]})
rows.append({"method": "enet_blend",     "RAE": m_enet["RAE"]})
comp_df = pd.DataFrame(rows).sort_values("RAE").reset_index(drop=True)
print(comp_df.round(4).to_string(index=False))

# Pick best strategy
best_strat = comp_df.iloc[0]["method"]
best_rae_v = comp_df.iloc[0]["RAE"]
print(f"\nBest strategy: {best_strat} (OOF RAE={best_rae_v:.4f})")

# Use ElasticNet as the final OOF (most principled)
oof = oof_enet


=== Delta Family Blend Comparison ===
                method    RAE
            enet_blend 0.2748
delta_similarity_tiers 0.2772
          delta_5tiers 0.2888
            delta_loso 0.3266
  multi_template_delta 0.3266
     delta_uncertainty 0.3268
      reverse_delta_ml 0.3269
         rank_weighted 0.3391
        simple_average 0.3577
              delta_ml 0.4164
    consensus_delta_ml 0.5173
    delta_chemprop_cpu 0.5585

Best strategy: enet_blend (OOF RAE=0.2748)


In [8]:
# --- Final test predictions ---
print("\nFitting final ElasticNet on all train OOFs...", flush=True)
meta_final = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
meta_final.fit(OOF_stack, y_tr)
te_preds = np.clip(meta_final.predict(TE_stack), y_tr.min()-0.5, y_tr.max()+0.5)

coef_df = pd.DataFrame({"model": names, "weight": meta_final.coef_}).sort_values("weight", ascending=False)
print("ElasticNet weights:")
print(coef_df[coef_df.weight.abs()>1e-6].to_string(index=False))
print(f"intercept={meta_final.intercept_:.4f}")

np.save(DATA_PROCESSED/"oof_delta_ensemble_blend.npy", oof)
np.save(DATA_PROCESSED/"te_oof_delta_ensemble_blend.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"109_delta_ensemble_blend.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb109 OOF RAE = {m_enet['RAE']:.4f} ***")


Fitting final ElasticNet on all train OOFs...


ElasticNet weights:


                 model    weight
          delta_5tiers  0.821799
delta_similarity_tiers  0.165359
     delta_uncertainty  0.139946
  multi_template_delta  0.062568
            delta_loso  0.055081
      reverse_delta_ml  0.054727
    delta_chemprop_cpu -0.001331
    consensus_delta_ml -0.095910
              delta_ml -0.142674
intercept=-0.2373
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\109_delta_ensemble_blend.csv
Test: min=3.06 med=4.97 max=6.64

*** nb109 OOF RAE = 0.2748 ***
